In [1]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import math
import random
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from peft import PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
SFT_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"

OUT_DIR = "/kaggle/working/phi3_strategyqa_calibration"
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
REPORTS_DIR = os.path.join(OUT_DIR, "reports")
CACHE_DIR = os.path.join(OUT_DIR, "cache")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

SEED = 42
EVAL_SEED = 42

# Number of samples for each split.
N_VALIDATION = 200   # for threshold calibration
N_TEST = 200         # for final evaluation

# Prompt variants.
PROMPT_VARIANTS = ["plain", "anchored"]

# If True, ask the model to reason before answering.
# This often changes confidence / commitment behavior.
USE_REASONING_COT = True

# Generate with deterministic decoding.
MAX_NEW_TOKENS = 24

# A little bit of calibration curve detail.
N_THRESHOLD_STEPS = 201
N_RELIABILITY_BINS = 10

# Whether to force loading the merged SFT adapter if available.
USE_SFT_ADAPTER = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

TOKENIZER = None
MODEL = None

# ============================================================
# UTILS
# ============================================================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def save_df(df, path):
    ensure_dir(os.path.dirname(path))
    df.to_csv(path, index=False)

def save_json(obj, path):
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass

def preview_text(s, max_len=220):
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")

def to_jsonable(x):
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return json.dumps(str(x), ensure_ascii=False)

def normalize_text(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())

# ============================================================
# ADVANCED PLOTTING UTILS
# ============================================================

def bar_plot(labels, values, title, path, ylabel="Value"):
    plt.figure(figsize=(9, 4))
    plt.bar(labels, values, color='#4C72B0')
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def line_plot(x, ys, labels, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(10, 5))
    for y, label in zip(ys, labels):
        plt.plot(x, y, marker="o", linewidth=1.6, label=label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def scatter(x, y, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(6.8, 5.2))
    plt.scatter(x, y, alpha=0.8, color='#55A868')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def histogram(values, title, path, xlabel="Value", bins=10):
    plt.figure(figsize=(8.8, 4.2))
    plt.hist(values, bins=bins, color='#DD8452', edgecolor='black', alpha=0.8)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def heatmap(mat, xlabels, ylabels, title, path, xlabel="", ylabel="", cmap="viridis"):
    plt.figure(figsize=(max(8, len(xlabels) * 0.55), max(5, len(ylabels) * 0.38)))
    im = plt.imshow(mat, aspect="auto", cmap=cmap)
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    plt.yticks(range(len(ylabels)), ylabels)
    for i in range(len(ylabels)):
        for j in range(len(xlabels)):
            val = mat[i, j]
            color = "white" if val < (mat.max() / 2) else "black"
            plt.text(j, i, str(int(val)), ha="center", va="center", color=color)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def hexbin_plot(x, y, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(8, 6))
    hb = plt.hexbin(x, y, gridsize=20, cmap='inferno', mincnt=1)
    cb = plt.colorbar(hb)
    cb.set_label('Sample Density')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def violin_split_plot(df, val_col, split_col, title, path, ylabel="Value"):
    categories = sorted(df[split_col].unique())
    data = [df[df[split_col] == cat][val_col].dropna().values for cat in categories]
    
    plt.figure(figsize=(8, 5))
    if any(len(d) > 0 for d in data):
        parts = plt.violinplot(data, positions=range(len(categories)), showmeans=True, showextrema=True)
        for pc in parts['bodies']:
            pc.set_facecolor('#4C72B0')
            pc.set_edgecolor('black')
            pc.set_alpha(0.6)
        plt.title(title)
        plt.xticks(range(len(categories)), categories)
        plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def correlation_heatmap(df, cols, title, path):
    df_numeric = df.copy()
    for col in cols:
        if df_numeric[col].dtype == bool or df_numeric[col].dtype == object:
            df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce')
    
    sub_df = df_numeric[cols].dropna()
    if len(sub_df) < 2: return
    
    corr = sub_df.corr().values
    plt.figure(figsize=(8, 6))
    im = plt.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    plt.colorbar(im)
    plt.xticks(range(len(cols)), cols, rotation=30, ha='right')
    plt.yticks(range(len(cols)), cols)
    for i in range(len(cols)):
        for j in range(len(cols)):
            color = 'black' if abs(corr[i, j]) < 0.5 else 'white'
            plt.text(j, i, f"{corr[i, j]:.2f}", ha='center', va='center', color=color)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def radar_chart(categories, values_dict, title, path):
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    
    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    
    plt.xticks(angles[:-1], categories, size=10)
    ax.set_rlabel_position(0)
    
    for label, values in values_dict.items():
        vals = list(values)
        vals += vals[:1]
        ax.plot(angles, vals, linewidth=2, linestyle='solid', label=label)
        ax.fill(angles, vals, alpha=0.25)
        
    plt.title(title, size=15, y=1.1)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def plot_reliability(rel_df, title, path):
    df = rel_df.dropna(subset=["avg_confidence", "accuracy"]).copy()
    if len(df) == 0:
        return
    plt.figure(figsize=(6.5, 5.5))
    plt.plot([0, 1], [0, 1], linestyle="--", color='gray', label="ideal")
    plt.plot(df["avg_confidence"], df["accuracy"], marker="o", color='#C44E52', linewidth=2, markersize=8, label="model")
    plt.xlabel("Average confidence")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

# ============================================================
# CACHE / SAMPLING
# ============================================================

def cached_indices(name, ds_len, n, seed=EVAL_SEED):
    ensure_dir(CACHE_DIR)
    n = min(n, ds_len)
    path = os.path.join(CACHE_DIR, f"{name}_n{n}_seed{seed}.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    rng = np.random.RandomState(seed)
    idx = rng.choice(ds_len, size=n, replace=False).tolist()
    with open(path, "w") as f:
        json.dump(idx, f)
    return idx

def sample_dataset(ds, name, n, seed=EVAL_SEED):
    idx = cached_indices(name, len(ds), n, seed=seed)
    return ds.select(idx), idx

# ============================================================
# PARSING / SCORING
# ============================================================

def extract_yes_no(text):
    if text is None:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span = span.lower()

    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()

    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()

    return None

def is_hedged(text):
    t = normalize_text(text)
    hedges = [
        "i think", "probably", "maybe", "possibly", "it seems", "likely",
        "uncertain", "not sure", "cannot determine", "can't determine", "depends"
    ]
    return any(h in t for h in hedges)

def explicit_answer_emitted(text):
    t = normalize_text(text)
    return bool(re.search(r"\b(yes|no)\b", t))

def commitment_confidence(text):
    t = normalize_text(text)
    score = 0.5
    if explicit_answer_emitted(t):
        score += 0.2
    if "<answer>" in t and "</answer>" in t:
        score += 0.1
    if is_hedged(t):
        score -= 0.2
    if len(t.split()) <= 4:
        score += 0.1
    if len(t.split()) > 20:
        score -= 0.1
    return max(0.0, min(1.0, score))

# ============================================================
# MODEL LOADING
# ============================================================

def load_tokenizer():
    global TOKENIZER
    if TOKENIZER is None:
        src = SFT_PATH if (USE_SFT_ADAPTER and os.path.exists(SFT_PATH)) else BASE_MODEL
        TOKENIZER = AutoTokenizer.from_pretrained(src)
        if TOKENIZER.pad_token is None:
            TOKENIZER.pad_token = TOKENIZER.eos_token
    return TOKENIZER

def load_model():
    global MODEL
    kwargs = {"torch_dtype": DTYPE}
    try:
        kwargs["attn_implementation"] = "eager"
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    except Exception:
        kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)

    model = model.to(DEVICE)
    model.eval()

    if USE_SFT_ADAPTER and os.path.exists(SFT_PATH) and HAS_PEFT:
        try:
            model = PeftModel.from_pretrained(model, SFT_PATH).merge_and_unload().to(DEVICE)
            model.eval()
        except Exception:
            pass

    try:
        model.config.use_cache = False
    except Exception:
        pass

    MODEL = model
    return model, load_tokenizer()

# ============================================================
# DATA
# ============================================================

def load_strategyqa_split(split_name, n, name_tag):
    ds = load_dataset("ChilleD/StrategyQA", split=split_name)
    ds, idx = sample_dataset(ds, name_tag, n)
    samples = []
    for i, s in enumerate(ds):
        samples.append({
            "sample_id": i,
            "source_index": idx[i],
            "question": s["question"],
            "gold": "yes" if bool(s["answer"]) else "no",
            "answer_bool": bool(s["answer"]),
        })
    return samples

# ============================================================
# PROMPTS
# ============================================================

def build_prompt(question, variant="plain"):
    if variant == "anchored":
        return (
            f"## Question: {question}\n"
            "You are a careful reasoning assistant.\n"
            "Decide whether the answer is yes or no.\n"
            "Think briefly if needed, then put ONLY yes or no inside <answer></answer>."
        )
    return (
        f"Question: {question}\n"
        "Answer only yes or no inside <answer></answer>."
    )

def build_cot_prompt(question, variant="plain"):
    if variant == "anchored":
        return (
            f"## Question: {question}\n"
            "You are a careful reasoning assistant.\n"
            "Think step by step inside <think></think>, then give ONLY yes or no inside <answer></answer>."
        )
    return (
        f"Question: {question}\n"
        "Think step by step inside <think></think>, then give ONLY yes or no inside <answer></answer>."
    )

# ============================================================
# GENERATION / LOGITS
# ============================================================

@torch.inference_mode()
def generate(prompt, max_new_tokens=MAX_NEW_TOKENS):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    out = MODEL.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=TOKENIZER.eos_token_id,
        eos_token_id=TOKENIZER.eos_token_id,
    )
    full_output = TOKENIZER.decode(out[0], skip_special_tokens=True)
    prompt_len = inputs["input_ids"].shape[1]
    completion = TOKENIZER.decode(out[0][prompt_len:], skip_special_tokens=True)
    return full_output, completion

def yes_no_token_scores(prompt):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        logits = MODEL(**inputs).logits[0, -1].float()
    probs = torch.softmax(logits, dim=-1)

    yes_ids = set()
    no_ids = set()
    for w in [" yes", " yes.", "yes", "Yes", " YES"]:
        enc = TOKENIZER.encode(w, add_special_tokens=False)
        if len(enc) == 1:
            yes_ids.add(enc[0])
    for w in [" no", " no.", "no", "No", " NO"]:
        enc = TOKENIZER.encode(w, add_special_tokens=False)
        if len(enc) == 1:
            no_ids.add(enc[0])

    yes_prob = float(sum(probs[i].item() for i in yes_ids)) if yes_ids else 0.0
    no_prob = float(sum(probs[i].item() for i in no_ids)) if no_ids else 0.0
    yes_no_mass = yes_prob + no_prob
    score = math.log((yes_prob + 1e-12) / (no_prob + 1e-12))
    return {
        "yes_prob": yes_prob,
        "no_prob": no_prob,
        "yes_no_mass": yes_no_mass,
        "log_odds": score,
    }

# ============================================================
# CALIBRATION
# ============================================================

def calibrate_threshold(samples, variant="plain", use_cot=False):
    scores = []
    labels = []

    for s in samples:
        prompt = build_cot_prompt(s["question"], variant) if use_cot else build_prompt(s["question"], variant)
        scores_dict = yes_no_token_scores(prompt)
        scores.append(scores_dict["log_odds"])
        labels.append(1 if s["gold"] == "yes" else 0)

    scores = np.array(scores, dtype=np.float64)
    labels = np.array(labels, dtype=np.int64)

    thresholds = np.quantile(scores, np.linspace(0.02, 0.98, N_THRESHOLD_STEPS))
    rows = []
    best = {"threshold": 0.0, "accuracy": -1.0}
    for t in thresholds:
        preds = (scores >= t).astype(np.int64)
        acc = float((preds == labels).mean())
        rows.append({"threshold": float(t), "accuracy": acc})
        if acc > best["accuracy"]:
            best = {"threshold": float(t), "accuracy": acc}

    return best, pd.DataFrame(rows), scores, labels

# ============================================================
# EVALUATION
# ============================================================

def evaluate_samples(samples, variant="plain", threshold=0.0, use_cot=False, split_name="test"):
    rows = []
    for s in samples:
        prompt = build_cot_prompt(s["question"], variant) if use_cot else build_prompt(s["question"], variant)
        full_output, completion = generate(prompt)
        raw_pred = extract_yes_no(completion)
        scores = yes_no_token_scores(prompt)
        calibrated_pred = "yes" if scores["log_odds"] >= threshold else "no"
        correct_raw = (raw_pred == s["gold"])
        correct_cal = (calibrated_pred == s["gold"])

        rows.append({
            "sample_id": s["sample_id"],
            "source_index": s["source_index"],
            "split": split_name,
            "variant": variant,
            "use_cot": bool(use_cot),
            "question": s["question"],
            "gold": s["gold"],
            "answer_bool": s["answer_bool"],
            "prompt": prompt,
            "full_output": full_output,
            "completion": completion,
            "raw_prediction": raw_pred,
            "calibrated_prediction": calibrated_pred,
            "yes_prob": scores["yes_prob"],
            "no_prob": scores["no_prob"],
            "yes_no_mass": scores["yes_no_mass"],
            "log_odds": scores["log_odds"],
            "commitment_score": commitment_confidence(completion),
            "hedged": bool(is_hedged(completion)),
            "explicit_answer_emitted": bool(explicit_answer_emitted(completion)),
            "raw_correct": bool(correct_raw),
            "calibrated_correct": bool(correct_cal),
            "completion_tokens": len(TOKENIZER(completion).input_ids),
            "prompt_tokens": len(TOKENIZER(prompt).input_ids),
        })
    return pd.DataFrame(rows)

# ============================================================
# RELIABILITY / BINS
# ============================================================

def reliability_bins(df, score_col="log_odds", label_col="gold", pred_col="calibrated_prediction", n_bins=N_RELIABILITY_BINS):
    if len(df) == 0:
        return pd.DataFrame()
    scores = df[score_col].to_numpy(dtype=np.float64)
    labels = (df[label_col].astype(str).str.lower() == "yes").astype(np.int64).to_numpy()

    probs = 1 / (1 + np.exp(-scores))
    bin_ids = np.clip((probs * n_bins).astype(int), 0, n_bins - 1)

    rows = []
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            rows.append({
                "bin": b,
                "count": 0,
                "avg_confidence": np.nan,
                "accuracy": np.nan,
                "yes_rate": np.nan,
            })
            continue
        rows.append({
            "bin": b,
            "count": int(mask.sum()),
            "avg_confidence": float(probs[mask].mean()),
            "accuracy": float((df.loc[mask, pred_col].astype(str).str.lower() == df.loc[mask, label_col].astype(str).str.lower()).mean()),
            "yes_rate": float(labels[mask].mean()),
        })
    return pd.DataFrame(rows)

def summarize_variant(df):
    if len(df) == 0:
        return {}
    return {
        "n": int(len(df)),
        "raw_accuracy": float(df["raw_correct"].mean()),
        "calibrated_accuracy": float(df["calibrated_correct"].mean()),
        "explicit_answer_rate": float(df["explicit_answer_emitted"].mean()),
        "hedged_rate": float(df["hedged"].mean()),
        "mean_commitment_score": float(df["commitment_score"].mean()),
        "mean_log_odds": float(df["log_odds"].mean()),
        "mean_yes_prob": float(df["yes_prob"].mean()),
        "mean_no_prob": float(df["no_prob"].mean()),
        "mean_prompt_tokens": float(df["prompt_tokens"].mean()),
        "mean_completion_tokens": float(df["completion_tokens"].mean()),
    }

# ============================================================
# MAIN
# ============================================================

def main():
    if not HAS_PEFT:
        print("PEFT not installed. If SFT_PATH is a merged model, the script can still run.")

    print("Loading model and tokenizer ...")
    load_model()

    print("Loading StrategyQA splits ...")
    val_samples = load_strategyqa_split("train", N_VALIDATION, "strategyqa_validation")
    test_samples = load_strategyqa_split("test", N_TEST, "strategyqa_test")

    all_rows = []
    threshold_rows = []
    cal_rows = []
    rel_rows = []

    calibration_info = {}
    for variant in PROMPT_VARIANTS:
        print(f"Calibrating threshold for variant={variant}, use_cot={USE_REASONING_COT} ...")
        best, threshold_df, scores, labels = calibrate_threshold(
            val_samples,
            variant=variant,
            use_cot=USE_REASONING_COT,
        )
        calibration_info[variant] = best
        threshold_df["variant"] = variant
        threshold_df["use_cot"] = USE_REASONING_COT
        threshold_rows.append(threshold_df)

        for s, y in zip(scores, labels):
            cal_rows.append({
                "variant": variant,
                "use_cot": USE_REASONING_COT,
                "log_odds": float(s),
                "label_yes": int(y),
                "prob_yes": float(1 / (1 + np.exp(-s))),
            })

    for variant in PROMPT_VARIANTS:
        threshold = calibration_info[variant]["threshold"]
        print(f"Evaluating variant={variant} with threshold={threshold:.4f} ...")
        df = evaluate_samples(test_samples, variant=variant, threshold=threshold, use_cot=USE_REASONING_COT, split_name="test")
        all_rows.append(df)

        rel_df = reliability_bins(df, score_col="log_odds", label_col="gold", pred_col="calibrated_prediction")
        rel_df["variant"] = variant
        rel_df["use_cot"] = USE_REASONING_COT
        rel_rows.append(rel_df)

    results_df = pd.concat(all_rows, ignore_index=True)
    threshold_df = pd.concat(threshold_rows, ignore_index=True)
    cal_df = pd.DataFrame(cal_rows)
    rel_df = pd.concat(rel_rows, ignore_index=True)

    save_df(results_df, os.path.join(CSV_DIR, "strategyqa_results.csv"))
    save_df(threshold_df, os.path.join(CSV_DIR, "threshold_search.csv"))
    save_df(cal_df, os.path.join(CSV_DIR, "calibration_scores.csv"))
    save_df(rel_df, os.path.join(CSV_DIR, "reliability_bins.csv"))

    summary_rows = []
    for variant in PROMPT_VARIANTS:
        sdf = results_df[results_df["variant"] == variant].copy()
        summ = summarize_variant(sdf)
        summ.update({
            "variant": variant,
            "threshold": calibration_info[variant]["threshold"],
            "val_accuracy": calibration_info[variant]["accuracy"],
        })
        summary_rows.append(summ)
    summary_df = pd.DataFrame(summary_rows)
    save_df(summary_df, os.path.join(CSV_DIR, "variant_summary.csv"))

    cm_raw = pd.DataFrame(0, index=["yes", "no"], columns=["yes", "no"])
    cm_cal = pd.DataFrame(0, index=["yes", "no"], columns=["yes", "no"])
    for _, r in results_df.iterrows():
        gold = str(r["gold"]).lower()
        raw = str(r["raw_prediction"]).lower() if pd.notna(r["raw_prediction"]) else None
        cal = str(r["calibrated_prediction"]).lower() if pd.notna(r["calibrated_prediction"]) else None
        if gold in cm_raw.index and raw in cm_raw.columns:
            cm_raw.loc[gold, raw] += 1
        if gold in cm_cal.index and cal in cm_cal.columns:
            cm_cal.loc[gold, cal] += 1
    save_df(cm_raw.reset_index().rename(columns={"index": "gold"}), os.path.join(CSV_DIR, "confusion_matrix_raw.csv"))
    save_df(cm_cal.reset_index().rename(columns={"index": "gold"}), os.path.join(CSV_DIR, "confusion_matrix_calibrated.csv"))

    for variant in PROMPT_VARIANTS:
        tdf = threshold_df[threshold_df["variant"] == variant].sort_values("threshold")
        if not tdf.empty:
            line_plot(
                tdf["threshold"].tolist(),
                [tdf["accuracy"].tolist()],
                ["val accuracy"],
                f"Threshold search: {variant}",
                os.path.join(PLOTS_DIR, f"threshold_search_{variant}.png"),
                xlabel="Threshold (log-odds)",
                ylabel="Validation accuracy",
            )

    for variant in PROMPT_VARIANTS:
        rdf = rel_df[rel_df["variant"] == variant].copy().sort_values("bin")
        if not rdf.empty:
            plot_reliability(
                rdf,
                f"Reliability diagram: {variant}",
                os.path.join(PLOTS_DIR, f"reliability_{variant}.png"),
            )

    for variant in PROMPT_VARIANTS:
        sdf = results_df[results_df["variant"] == variant]
        if sdf.empty: continue
        
        histogram(
            sdf["log_odds"].tolist(),
            f"Log-odds distribution: {variant}",
            os.path.join(PLOTS_DIR, f"logodds_hist_{variant}.png"),
            xlabel="log(yes/no)",
            bins=15,
        )
        histogram(
            sdf["commitment_score"].tolist(),
            f"Commitment score distribution: {variant}",
            os.path.join(PLOTS_DIR, f"commitment_hist_{variant}.png"),
            xlabel="Commitment score",
            bins=10,
        )
        histogram(
            sdf["completion_tokens"].tolist(),
            f"Completion length distribution: {variant}",
            os.path.join(PLOTS_DIR, f"completion_length_{variant}.png"),
            xlabel="Completion tokens",
            bins=15,
        )

        bar_plot(
            ["raw", "calibrated"],
            [float(sdf["raw_correct"].mean()), float(sdf["calibrated_correct"].mean())],
            f"Accuracy: {variant}",
            os.path.join(PLOTS_DIR, f"accuracy_bar_{variant}.png"),
            ylabel="Accuracy",
        )
        bar_plot(
            ["explicit", "hedged"],
            [float(sdf["explicit_answer_emitted"].mean()), float(sdf["hedged"].mean())],
            f"Commitment / hedging: {variant}",
            os.path.join(PLOTS_DIR, f"commitment_bar_{variant}.png"),
            ylabel="Rate",
        )

        hexbin_plot(
            sdf["log_odds"].tolist(),
            sdf["commitment_score"].tolist(),
            f"Density: Log-odds vs Commitment ({variant})",
            os.path.join(PLOTS_DIR, f"logodds_vs_commitment_hexbin_{variant}.png"),
            xlabel="Log-odds",
            ylabel="Commitment Score"
        )
        
        violin_split_plot(
            sdf,
            "log_odds",
            "gold",
            f"Log-Odds Confidence Split by Gold Label ({variant})",
            os.path.join(PLOTS_DIR, f"logodds_violin_by_class_{variant}.png"),
            ylabel="Log-Odds (Yes/No)"
        )
        
        correlation_heatmap(
            sdf,
            ["completion_tokens", "log_odds", "commitment_score", "raw_correct", "calibrated_correct"],
            f"Cross-Metric Correlation Matrix ({variant})",
            os.path.join(PLOTS_DIR, f"metric_correlations_{variant}.png")
        )

    overall_summary = []
    radar_data = {}
    radar_categories = ["Raw Acc", "Calibrated Acc", "Commitment", "Explicit Rate", "Un-Hedged Rate"]
    
    for variant in PROMPT_VARIANTS:
        sdf = results_df[results_df["variant"] == variant]
        if not sdf.empty:
            raw_acc = float(sdf["raw_correct"].mean())
            cal_acc = float(sdf["calibrated_correct"].mean())
            exp_rate = float(sdf["explicit_answer_emitted"].mean())
            hedged_rate = float(sdf["hedged"].mean())
            commit = float(sdf["commitment_score"].mean())
            
            overall_summary.append({
                "variant": variant,
                "raw_accuracy": raw_acc,
                "calibrated_accuracy": cal_acc,
                "explicit_answer_rate": exp_rate,
                "hedged_rate": hedged_rate,
                "mean_commitment_score": commit,
                "mean_log_odds": float(sdf["log_odds"].mean()),
                "threshold": calibration_info[variant]["threshold"],
                "validation_accuracy": calibration_info[variant]["accuracy"],
            })
            
            radar_data[variant] = [raw_acc, cal_acc, commit, exp_rate, 1.0 - hedged_rate]

    if radar_data:
        radar_chart(
            radar_categories,
            radar_data,
            "Prompt Variant Multi-Dimensional Performance Fingerprint",
            os.path.join(PLOTS_DIR, "variant_performance_radar.png")
        )

    overall_df = pd.DataFrame(overall_summary)
    save_df(overall_df, os.path.join(CSV_DIR, "overall_summary.csv"))

    md = ["# StrategyQA commitment / calibration experiment\n"]
    md.append(f"- Model: `{BASE_MODEL}`")
    md.append(f"- Validation samples: {N_VALIDATION}")
    md.append(f"- Test samples: {N_TEST}")
    md.append(f"- Prompt variants: {', '.join(PROMPT_VARIANTS)}")
    md.append(f"- Reasoning CoT enabled: {USE_REASONING_COT}")
    md.append("\n## Overall results\n")
    md.append("| Variant | Raw acc | Calibrated acc | Explicit answer rate | Hedged rate | Mean commitment | Threshold | Val acc |")
    md.append("|---|---:|---:|---:|---:|---:|---:|---:|")
    for _, r in overall_df.iterrows():
        md.append(
            f"| {r['variant']} | {r['raw_accuracy']:.3f} | {r['calibrated_accuracy']:.3f} | {r['explicit_answer_rate']:.3f} | {r['hedged_rate']:.3f} | {r['mean_commitment_score']:.3f} | {r['threshold']:.4f} | {r['validation_accuracy']:.3f} |"
        )

    md.append("\n## Interpretation hints\n")
    md.append("- High explicit-answer rate but lower raw accuracy suggests the model commits, but not always correctly.")
    md.append("- A calibrated threshold that improves accuracy suggests the model has useful confidence information.")
    md.append("- High hedging with decent calibrated accuracy suggests the model knows but hesitates.")
    md.append("- Comparing plain vs anchored prompts shows whether the delimiter changes yes/no calibration.")

    with open(os.path.join(REPORTS_DIR, "report.md"), "w") as f:
        f.write("\n".join(md))

    save_json({
        "config": {
            "BASE_MODEL": BASE_MODEL,
            "SFT_PATH": SFT_PATH,
            "N_VALIDATION": N_VALIDATION,
            "N_TEST": N_TEST,
            "PROMPT_VARIANTS": PROMPT_VARIANTS,
            "USE_REASONING_COT": USE_REASONING_COT,
            "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
            "N_THRESHOLD_STEPS": N_THRESHOLD_STEPS,
            "N_RELIABILITY_BINS": N_RELIABILITY_BINS,
        },
        "calibration": calibration_info,
        "summary": overall_summary,
    }, os.path.join(REPORTS_DIR, "summary.json"))

    print("\n===== SUMMARY =====")
    if not overall_df.empty:
        print(overall_df.to_string(index=False))
    print("\nSaved to:")
    print(f"- {OUT_DIR}")
    print(f"- CSVs: {CSV_DIR}")
    print(f"- Plots: {PLOTS_DIR}")
    print(f"- Reports: {REPORTS_DIR}")

    free_memory(MODEL)

if __name__ == "__main__":
    main()

Loading model and tokenizer ...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

data/train-00000-of-00001-506370352f6228(…):   0%|          | 0.00/369k [00:00<?, ?B/s]

data/test-00000-of-00001-bae602f3ee37f4c(…):   0%|          | 0.00/161k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1603 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/687 [00:00<?, ? examples/s]

Calibrating threshold for variant=plain, use_cot=True ...
Calibrating threshold for variant=anchored, use_cot=True ...
Evaluating variant=plain with threshold=-1.5064 ...
Evaluating variant=anchored with threshold=-1.6677 ...

===== SUMMARY =====
 variant  raw_accuracy  calibrated_accuracy  explicit_answer_rate  hedged_rate  mean_commitment_score  mean_log_odds  threshold  validation_accuracy
   plain         0.185                 0.51                 0.260          0.0                  0.571      -1.364594  -1.506415                0.590
anchored         0.130                 0.55                 0.215          0.0                  0.558      -2.067092  -1.667671                0.595

Saved to:
- /kaggle/working/phi3_strategyqa_calibration
- CSVs: /kaggle/working/phi3_strategyqa_calibration/csv
- Plots: /kaggle/working/phi3_strategyqa_calibration/plots
- Reports: /kaggle/working/phi3_strategyqa_calibration/reports
